# QuantumJam 2026 · Curva de energía de $\mathrm{H_4}$ con VQE

Cadena lineal de 4 H, espaciado uniforme $d$, base STO-3G. Enunciado en `docs/`.

## 0. Instalación

In [ ]:
%pip install -q qiskit==2.5.2 qiskit-aer==0.17.2 pyscf==2.14.0 numpy scipy pandas matplotlib

## 0.1 Contrato compartido

Nombres, firmas y convenciones que usa todo el notebook. Cada sección reemplaza su `NotImplementedError`; cambios a este contrato se avisan en el issue.

In [ ]:
# Contrato compartido: nombres, firmas y convenciones que usa todo el notebook.
# Cada sección implementa su parte; nadie cambia esto sin avisar en el issue correspondiente.
import numpy as np
import pandas as pd
from qiskit import QuantumCircuit, transpile
from qiskit.quantum_info import SparsePauliOp
from qiskit.transpiler import CouplingMap

# --- Grilla y geometría -------------------------------------------------------
BASIS = "sto-3g"
D = np.round(np.concatenate([np.arange(0.6, 1.75, 0.1), np.arange(1.8, 3.01, 0.2)]), 2)  # Å, 19 puntos
STRETCHED = 1.8    # Å: desde acá hay que ganarle a CCSD
DISSOCIATED = 3.0  # Å: D_e = E(3,0) - E(R_e)
CHEM_ACC = 1.6e-3  # Ha


def h4_geometry(d: float) -> str:
    """Cadena lineal de 4 H sobre el eje z, separados por d Å (formato PySCF)."""
    return "; ".join(f"H 0 0 {i * d:.6f}" for i in range(4))


# --- Convención de qubits (Jordan-Wigner) --------------------------------------
# Orbitales espaciales MO de RHF, p = 0..3 en orden de energía.
# qubit p     = espín-orbital (p, α)
# qubit 4 + p = espín-orbital (p, β)
# |1> = ocupado. Qiskit es little-endian: el label de un estado se lee q7 ... q0.
N_ORB, N_QUBITS, N_ALPHA, N_BETA = 4, 8, 2, 2
HF_BITSTRING = "00110011"  # q0, q1 (α 0,1) y q4, q5 (β 0,1) ocupados


def alpha_qubit(p: int) -> int:
    return p


def beta_qubit(p: int) -> int:
    return N_ORB + p


# --- Transpilación y conteo de cz (igual para todos) ---------------------------
BASIS_GATES = ["cz", "sx", "x", "rz"]


def transpile_line(qc: QuantumCircuit) -> QuantumCircuit:
    return transpile(qc, basis_gates=BASIS_GATES, coupling_map=CouplingMap.from_line(qc.num_qubits),
                     optimization_level=3, seed_transpiler=0)


def cz_count(qc: QuantumCircuit) -> int:
    return transpile_line(qc).count_ops().get("cz", 0)


# --- Firmas que implementa cada sección ----------------------------------------
# Tablas compartidas (energías totales en Ha, con repulsión nuclear):
#   refs   : DataFrame  d, e_hf, e_ccsd, e_fci           (#1)
#   params : dict       d -> np.ndarray θ*               (#5)
#   vqe    : DataFrame  d, energy, sigma, shots, cz      (#6)

def references(d: float) -> dict:
    """#1. {'d', 'e_hf', 'e_ccsd', 'e_fci'}."""
    raise NotImplementedError


def qubit_hamiltonian(d: float) -> tuple[SparsePauliOp, float]:
    """#2. (H electrónico como SparsePauliOp de 8 qubits en la convención de arriba, e_nuc)."""
    raise NotImplementedError


def build_ansatz() -> QuantumCircuit:
    """#4. Circuito parametrizado de 8 qubits que arranca en HF_BITSTRING."""
    raise NotImplementedError


## 1. Referencias clásicas (HF, CCSD, FCI)

## 2. Hamiltoniano de qubits

## 3. Ansatz

### Decisión (#4)

Elegimos un **2-QUpCCGSD podado**: excitaciones de pares $(p_\uparrow p_\downarrow)\to(q_\uparrow q_\downarrow)$ y simples $p\to q$ (mismo ángulo en α y β), como excitaciones de **qubit** (sin cadenas Z), en dos capas, con 19 de las 24 excitaciones.

Comparación con statevector (error vs. FCI en mHa a 0,9 / 1,8 / 3,0 Å; `cz` tras transpilar a `cz, sx, x, rz` en línea):

| Ansatz | `cz` | Parámetros | Error (mHa) |
|---|---|---|---|
| UCCSD (fermiónico) | 1277 | 26 | 0,03 / 3,3 / 0,46 |
| UCCSD de qubits | 1247 | 26 | 0,003 / 3,0 / 0,21 |
| 1-UpCCGSD | 474 | 12 | 14 / 11 / 0,29 |
| 2-UpCCGSD (fermiónico) | 982 | 24 | exacto |
| 2-QUpCCGSD (Givens) | 590 | 24 | exacto |
| **2-QUpCCGSD podado (elegido)** | **465** | **19** | **≤ 0,13 en toda la grilla** |
| 2-QUpCCGSD podado más agresivo | 411 | 17 | ≤ 0,5 (en 6 distancias) |
| *CCSD (referencia)* | – | – | 0,007 / −9,1 / −23 |

Por qué:

- **Llega a FCI en la zona estirada.** UCCSD falla ahí (3,3 mHa a 1,8 Å, fuera de precisión química) por la misma razón que CCSD: el estado se vuelve multirreferencia. Las excitaciones generalizadas de pares sí lo capturan.
- **Conserva N = 4 y S_z = 0**: cada excitación mueve electrones dentro del mismo bloque de espín. Lo verificamos numéricamente abajo, y sirve para el argumento de confiabilidad (#8).
- **Menos `cz`**: cada excitación es una rotación de Givens entre dos estados de la base, implementada con un multiplexor de RY en código Gray (14 CX por doble) en lugar de `PauliEvolutionGate` (837 → 590 `cz`). Después podamos de a una las excitaciones de menor ángulo mientras el error siguiera < 0,1 mHa en 6 distancias (590 → 465 `cz`).

Qué no funcionó o descartamos:

- UCCSD y 1-UpCCGSD: no llegan a precisión química en la zona estirada.
- Sintetizar las excitaciones con `PauliEvolutionGate`: correcto pero caro en `cz`.
- Hardware-efficient (`efficient_su2`): descartado sin medirlo a fondo porque no conserva N ni S_z.
- La versión de 17 excitaciones (411 `cz`) queda como alternativa si los extras con ruido necesitan menos compuertas, a costa de errores de hasta 0,5 mHa.

Limitaciones:

- **El paisaje de energía tiene mínimos locales.** Con un solo arranque, algunas distancias quedan a 1 mHa o hasta decenas de mHa; con 30 arranques el mejor llega a ≤ 0,003 mHa. Con los 10 arranques de la validación de abajo, el peor punto queda entre 0,04 y 0,13 mHa según la corrida (diferencias numéricas mínimas cambian a qué mínimo cae cada arranque). La optimización (#5) tiene que usar multi-start.
- **Contaminación de espín en la zona más estirada.** Conservar N y S_z no garantiza S² = 0: las excitaciones de qubit no están adaptadas al espín. Hasta 1,8 Å el estado optimizado tiene ⟨S²⟩ ≤ 1e-5 (singlete puro en la práctica), pero desde 2,0 Å aparece una mezcla chica con el triplete: ⟨S²⟩ de 1e-4 a 3e-2 según la distancia y la corrida (el peor visto, 0,03 a 2,4 Å, es ~1,5 % de peso de triplete; depende de a qué mínimo local cae la optimización). La energía no lo detecta porque la brecha con el primer estado excitado se achica a 0,5 mHa a 3,0 Å: mezclar un poco de triplete casi no cuesta energía. Por eso la tabla de abajo reporta ⟨S²⟩ y la brecha; #8 tiene que incluirlo en el argumento de confiabilidad, y #5 puede eliminarlo con una penalización λ⟨S²⟩ en la optimización.
- **465 `cz` es mucho para hardware ruidoso.** Con 0,1 % de error por `cz`, la probabilidad de al menos una falla es 1 − 0,999⁴⁶⁵ ≈ 37 %. Para el desafío central (sin ruido) no importa; para los extras con ruido conviene la versión podada más agresiva o menos qubits (#14).

La validación de abajo usa un **stand-in** del Hamiltoniano (#2): H restringido a los 36 estados con N = 4 y S_z = 0, construido con PySCF. Verificamos que la convención de signos de PySCF coincide exactamente con la de Jordan-Wigner de la celda 0.1 (ver comentario en #4). Cuando #2 esté, la energía se puede recalcular con `qubit_hamiltonian`.


In [ ]:
# Ansatz elegido (#4): 2-QUpCCGSD podado, 19 excitaciones de qubit como rotaciones de Givens.
import itertools
from qiskit.circuit import ParameterVector


def givens_single(qc: QuantumCircuit, t, i: int, k: int) -> None:
    """exp(θ(Q†_k Q_i − h.c.)): rota |q_i=1, q_k=0⟩ hacia |q_i=0, q_k=1⟩."""
    qc.cx(k, i)
    qc.cry(2 * t, i, k)
    qc.cx(k, i)


def _ucry(qc: QuantumCircuit, t, controls: list, target: int, pattern: tuple) -> None:
    """RY(2θ) en target solo cuando controls == pattern. Multiplexor Gray: 2^n CX."""
    n = 2 ** len(controls)
    gray = [m ^ (m >> 1) for m in range(n)]
    idx = sum(b << c for c, b in enumerate(pattern))
    for m in range(n):
        qc.ry(2 * t * (-1) ** bin(idx & gray[m]).count("1") / n, target)
        qc.cx(controls[(gray[m] ^ gray[(m + 1) % n]).bit_length() - 1], target)


def givens_double(qc: QuantumCircuit, t, i: int, j: int, k: int, l: int) -> None:
    """exp(θ(Q†_k Q†_l Q_j Q_i − h.c.)): rota |ij=11, kl=00⟩ hacia |ij=00, kl=11⟩ (14 CX)."""
    for m in (i, j, l):
        qc.cx(k, m)
    _ucry(qc, t, [i, j, l], k, (1, 1, 0))
    for m in (l, j, i):
        qc.cx(k, m)


# (capa, tipo, p, q). "D": par (p↑, p↓) → (q↑, q↓). "S": simple p → q, en α y en β con el mismo θ.
ANSATZ_OPS = [
    (0, "D", 0, 2), (0, "D", 1, 2), (0, "D", 1, 3), (0, "D", 2, 3),
    (0, "S", 0, 1), (0, "S", 0, 3), (0, "S", 1, 2), (0, "S", 1, 3), (0, "S", 2, 3),
    (1, "D", 0, 1), (1, "D", 0, 3), (1, "D", 1, 2), (1, "D", 1, 3), (1, "D", 2, 3),
    (1, "S", 0, 2), (1, "S", 0, 3), (1, "S", 1, 2), (1, "S", 1, 3), (1, "S", 2, 3),
]


def build_ansatz() -> QuantumCircuit:
    """#4. Circuito de 8 qubits que arranca en HF_BITSTRING; un parámetro por excitación."""
    th = ParameterVector("θ", len(ANSATZ_OPS))
    qc = QuantumCircuit(N_QUBITS, name="QUpCCGSD-19")
    for q, bit in enumerate(reversed(HF_BITSTRING)):
        if bit == "1":
            qc.x(q)
    for t, (_, kind, p, q) in zip(th, ANSATZ_OPS):
        if kind == "D":
            givens_double(qc, t, alpha_qubit(p), beta_qubit(p), alpha_qubit(q), beta_qubit(q))
        else:
            givens_single(qc, t, alpha_qubit(p), alpha_qubit(q))
            givens_single(qc, t, beta_qubit(p), beta_qubit(q))
    return qc


ansatz = build_ansatz()
print(f"{ansatz.name}: {ansatz.num_parameters} parámetros, {cz_count(ansatz)} cz (transpilado a cz/sx/x/rz en línea)")


In [ ]:
# Validación del ansatz en toda la grilla, con statevector (permitido para elegir parámetros y ansatz).
# Stand-in del Hamiltoniano mientras no esté #2: H restringido al sector N=4, S_z=0 (36 estados),
# armado con PySCF. Verificamos en #4 que la convención de signos de PySCF coincide con la de la celda 0.1.
from pyscf import ao2mo, fci, gto, scf
from pyscf.fci import cistring, spin_op
from scipy.optimize import minimize
from qiskit.quantum_info import Statevector

_STRINGS = cistring.make_strings(range(N_ORB), N_ALPHA)
SECTOR = np.array([int(a) | (int(b) << N_ORB) for a in _STRINGS for b in _STRINGS])  # índices de qubits


def sector_hamiltonian(d: float) -> tuple[np.ndarray, float]:
    """H total (con e_nuc) en el sector de 36 estados y E_FCI."""
    mol = gto.M(atom=h4_geometry(d), basis=BASIS, unit="Angstrom", verbose=0)
    mf = scf.RHF(mol).run()
    h1 = mf.mo_coeff.T @ mf.get_hcore() @ mf.mo_coeff
    eri = ao2mo.restore(1, ao2mo.full(mol, mf.mo_coeff), N_ORB)
    h2 = fci.direct_spin1.absorb_h1e(h1, eri, N_ORB, (N_ALPHA, N_BETA), 0.5)
    H = np.column_stack([fci.direct_spin1.contract_2e(h2, e.reshape(6, 6), N_ORB, (N_ALPHA, N_BETA)).ravel()
                         for e in np.eye(len(SECTOR))])
    return H + mol.energy_nuc() * np.eye(len(SECTOR)), fci.FCI(mf).kernel()[0]


def _qubit_excitation(create: tuple, annihilate: tuple) -> np.ndarray:
    """K = T − T† en el sector, con T = Q†...Q... (operadores de qubit, sin cadenas Z)."""
    def q(j, dag):
        lab = lambda p: "I" * (N_QUBITS - j - 1) + p + "I" * j
        return SparsePauliOp([lab("X"), lab("Y")], [0.5, -0.5j if dag else 0.5j])
    T = SparsePauliOp("I" * N_QUBITS)
    for c in create:
        T = T @ q(c, True)
    for a in annihilate:
        T = T @ q(a, False)
    M = (T - T.adjoint()).simplify().to_matrix(sparse=True)[SECTOR][:, SECTOR].toarray()
    return M.real


class SectorAnsatz:
    """El mismo ansatz simulado en el sector de 36 estados, con gradiente adjunto exacto (rápido)."""

    def __init__(self, ops):
        self.groups = []
        for _, kind, p, q in ops:
            if kind == "D":
                self.groups.append([_qubit_excitation((alpha_qubit(q), beta_qubit(q)), (beta_qubit(p), alpha_qubit(p)))])
            else:
                self.groups.append([_qubit_excitation((alpha_qubit(q),), (alpha_qubit(p),)),
                                    _qubit_excitation((beta_qubit(q),), (beta_qubit(p),))])
        self.n = len(self.groups)
        self.hf = (SECTOR == int(HF_BITSTRING, 2)).astype(float)

    def _U(self, k, t):  # cada generador cumple K³ = −K, así que exp(tK) = 1 + sin(t)K + (1 − cos(t))K²
        I = np.eye(len(SECTOR))
        U = I
        for K in self.groups[k]:
            U = (I + np.sin(t) * K + (1 - np.cos(t)) * K @ K) @ U
        return U

    def state(self, x):
        v = self.hf
        for k in range(self.n):
            v = self._U(k, x[k]) @ v
        return v

    def energy_grad(self, x, H):
        Us = [self._U(k, x[k]) for k in range(self.n)]
        v = self.hf
        for U in Us:
            v = U @ v
        lam = H @ v
        e, g = v @ lam, np.empty(self.n)
        for k in reversed(range(self.n)):
            g[k] = 2 * lam @ (sum(self.groups[k]) @ v)
            v, lam = Us[k].T @ v, Us[k].T @ lam
        return e, g


sector_ansatz = SectorAnsatz(ANSATZ_OPS)

# El circuito real y la simulación en el sector tienen que dar el mismo estado.
_x = np.random.default_rng(0).normal(size=ansatz.num_parameters)
_full = Statevector(ansatz.assign_parameters(_x)).data
assert np.isclose(np.linalg.norm(_full[SECTOR]), 1), "el ansatz se sale del sector N=4, S_z=0"
assert np.isclose(abs(np.vdot(_full[SECTOR], sector_ansatz.state(_x))), 1), "circuito ≠ simulación en el sector"

# El paisaje de energía tiene mínimos locales (algunos arranques quedan a decenas de mHa): por eso
# probamos varios arranques por distancia (cero, el óptimo de la distancia anterior y 8 al azar).
rng = np.random.default_rng(0)
rows, x0 = [], np.zeros(sector_ansatz.n)
for d in D:
    H, e_fci = sector_hamiltonian(d)
    starts = [np.zeros(sector_ansatz.n), x0] + [rng.normal(scale=0.7, size=sector_ansatz.n) for _ in range(8)]
    best = min((minimize(sector_ansatz.energy_grad, s, args=(H,), jac=True, method="BFGS", options={"gtol": 1e-10})
                for s in starts), key=lambda r: r.fun)
    x0 = best.x
    s2 = spin_op.spin_square0(sector_ansatz.state(best.x).reshape(6, 6), N_ORB, (N_ALPHA, N_BETA))[0]
    gap = np.diff(np.linalg.eigvalsh(H)[:2])[0]
    rows.append({"d": d, "e_ansatz": best.fun, "e_fci": e_fci, "err_mHa": (best.fun - e_fci) * 1e3,
                 "S2": s2, "gap_mHa": gap * 1e3})
ansatz_check = pd.DataFrame(rows)
print(f"error máximo vs. FCI en la grilla: {ansatz_check.err_mHa.abs().max():.3f} mHa "
      f"(zona estirada: {ansatz_check[ansatz_check.d >= STRETCHED].err_mHa.abs().max():.3f} mHa); "
      f"⟨S²⟩ máximo: {ansatz_check.S2.max():.1e} (singlete puro = 0, triplete = 2)")
ansatz_check


## 4. VQE

## 5. Resultados: curva, $R_e$, $D_e$ y tabla (energía, $\sigma$, shots, `cz`)

## 6. Zona estirada ($d \geq 1{,}8$ Å) vs. CCSD

## 7. Decisiones, qué no funcionó y limitaciones